# GEANT h96 partial DG-STMTL comparison (reporting only)

This notebook verifies the pinned reporting package, reads exactly 11 explicitly named `run.json` manifests from Drive, and creates a non-overwriting validation report. It does not open arrays, predictions, checkpoint bytes, or test-split artifacts; it does not train or run inference.

The eight original GNN runs intentionally omit original GNN Topology seed 42. The report remains partial: n=2 for Topology, n=3 for Identity, Shuffled, and DG-STMTL-based. Upload the source ZIP and its manifest. The notebook asks for the existing Review18 recovery backup root because its session UUID is not fixed in this notebook.

In [ ]:
from pathlib import Path
import hashlib, io, json, os, shutil, stat, subprocess, sys, uuid, zipfile
from google.colab import files

EXPECTED_SOURCE_ZIP_SHA256 = "b202922d1dc2060d18c341dcd53b57cd2ec89b2a71f33c2fe40b4492c9372f60"
EXPECTED_SOURCE_MANIFEST_SHA256 = "48645bad2685ed218789ff7085a788727836f78ce1d3c8bfebf1d5ca5de639d7"
EXPECTED_MEMBERS = {"README.md", "phase9_geant/compare_dg_stmtl_h96.py"}
SESSION_ID = uuid.uuid4().hex
STAGE = Path('/content') / f'geant_h96_partial_dg_report_v1_{SESSION_ID}'
if STAGE.exists() or STAGE.is_symlink(): raise FileExistsError(f'Refusing existing staging path: {STAGE}')
def sha256_bytes(data): return hashlib.sha256(data).hexdigest()

print('Upload exactly the reporting source ZIP and source_snapshot_manifest.json. Filename suffixes are ignored; SHA-256 identifies each file.')
uploads = files.upload()
wanted = {EXPECTED_SOURCE_ZIP_SHA256: 'source_zip', EXPECTED_SOURCE_MANIFEST_SHA256: 'source_manifest'}
matched = {}
for uploaded_name, uploaded_bytes in uploads.items():
    digest = sha256_bytes(uploaded_bytes)
    if digest not in wanted: raise RuntimeError(f'FAIL unrecognized upload by SHA-256: {uploaded_name} ({digest})')
    role = wanted[digest]
    if role in matched: raise RuntimeError(f'FAIL duplicate upload for {role}')
    matched[role] = (uploaded_name, uploaded_bytes)
if set(matched) != {'source_zip', 'source_manifest'}: raise RuntimeError('FAIL both pinned package files are required')
source_manifest = json.loads(matched['source_manifest'][1].decode('utf-8'))
if source_manifest.get('bundle_id') != 'geant-h96-partial-dg-stmtl-report-v1': raise RuntimeError('FAIL source manifest bundle identity mismatch')
if source_manifest.get('archive_sha256') != EXPECTED_SOURCE_ZIP_SHA256: raise RuntimeError('FAIL archive pin mismatch')
if {item.get('path') for item in source_manifest.get('members', [])} != EXPECTED_MEMBERS: raise RuntimeError('FAIL manifest member inventory mismatch')
archive_bytes = matched['source_zip'][1]
if sha256_bytes(archive_bytes) != EXPECTED_SOURCE_ZIP_SHA256: raise RuntimeError('FAIL source ZIP SHA-256 mismatch')
STAGE.mkdir(parents=False, exist_ok=False)
extract_root = STAGE / 'approved_reporting_source'
extract_root.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(archive_bytes), 'r') as archive:
    infos = archive.infolist()
    names = [item.filename for item in infos]
    if len(names) != len(set(names)) or set(names) != EXPECTED_MEMBERS: raise RuntimeError('FAIL ZIP inventory mismatch or duplicate member')
    for item in infos:
        name = item.filename.replace('\\', '/')
        mode = (item.external_attr >> 16) & 0xFFFF
        if name.startswith('/') or any(part in ('', '.', '..') for part in name.split('/')): raise RuntimeError(f'FAIL unsafe ZIP path: {item.filename}')
        if stat.S_ISLNK(mode) or item.is_dir(): raise RuntimeError(f'FAIL symlink or unexpected directory: {item.filename}')
        expected = next(entry for entry in source_manifest['members'] if entry['path'] == name)
        payload = archive.read(item)
        if len(payload) != expected['bytes'] or sha256_bytes(payload) != expected['sha256']: raise RuntimeError(f'FAIL member integrity mismatch: {name}')
        destination = extract_root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() or destination.is_symlink(): raise FileExistsError(f'Refusing existing extracted file: {destination}')
        with destination.open('xb') as stream: stream.write(payload)
COMPARE = extract_root / 'phase9_geant' / 'compare_dg_stmtl_h96.py'
script_hash = next(entry['sha256'] for entry in source_manifest['members'] if entry['path'] == 'phase9_geant/compare_dg_stmtl_h96.py')
if not COMPARE.is_file() or sha256_bytes(COMPARE.read_bytes()) != script_hash: raise RuntimeError('FAIL extracted comparator identity mismatch')
help_result = subprocess.run([sys.executable, str(COMPARE), '--help'], text=True, capture_output=True, check=False)
if help_result.returncode != 0 or '--allow-incomplete-gnn-matrix' not in help_result.stdout: raise RuntimeError(f'FAIL comparator CLI check: {help_result.stderr}')
print({'stage': 'package_verification', 'status': 'PASS', 'zip_sha256': EXPECTED_SOURCE_ZIP_SHA256, 'manifest_sha256': EXPECTED_SOURCE_MANIFEST_SHA256, 'members': sorted(EXPECTED_MEMBERS), 'training': False, 'test_access': False})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_BASE = Path('/content/drive/MyDrive/SDNResearch').resolve(strict=True)
if not DRIVE_BASE.is_dir(): raise RuntimeError('FAIL expected SDNResearch Drive directory unavailable')
MATRIX3_RUNS = DRIVE_BASE / 'geant_h96_matrix3_5fe371f8aab74f42983569579477c328' / 'runs'
DG_RUNS = DRIVE_BASE / 'geant_dg_stmtl_h96_v1_review3' / 'runs'
review18_text = input('Enter the exact existing Review18 recovery backup ROOT (folder containing runs/): ').strip()
review18_entered = Path(os.path.abspath(review18_text))
if review18_entered.is_symlink(): raise RuntimeError('FAIL Review18 root is a symlink')
REVIEW18_ROOT = review18_entered.resolve(strict=True)
if REVIEW18_ROOT.parent != DRIVE_BASE or not REVIEW18_ROOT.name.startswith('geant_h96_matrix3_recovery_review18_'): raise RuntimeError('FAIL Review18 root outside approved Drive parent or unexpected name')
REVIEW18_RUNS = REVIEW18_ROOT / 'runs'
if any(not path.is_dir() or path.is_symlink() for path in (MATRIX3_RUNS, DG_RUNS, REVIEW18_RUNS)): raise RuntimeError('FAIL expected read-only backup runs directory missing or redirected')
def manifest_path(root, run_name):
    run_dir = root / run_name
    path = run_dir / 'run.json'
    if run_dir.is_symlink() or path.is_symlink() or not path.is_file() or path.resolve(strict=True) != Path(os.path.abspath(path)): raise RuntimeError(f'FAIL missing or redirected manifest: {path}')
    return path
gnn_manifest_paths = [
    manifest_path(MATRIX3_RUNS, 'h96_topology_seed_123'),
    manifest_path(MATRIX3_RUNS, 'h96_topology_seed_2026'),
    manifest_path(MATRIX3_RUNS, 'h96_identity_seed_42'),
    manifest_path(MATRIX3_RUNS, 'h96_identity_seed_123'),
    manifest_path(MATRIX3_RUNS, 'h96_identity_seed_2026'),
    manifest_path(MATRIX3_RUNS, 'h96_shuffled_seed_42'),
    manifest_path(REVIEW18_RUNS, 'h96_shuffled_seed_123'),
    manifest_path(REVIEW18_RUNS, 'h96_shuffled_seed_2026'),
]
dg_manifest_paths = [manifest_path(DG_RUNS, f'dg_stmtl_topology_seed_{seed}') for seed in (42, 123, 2026)]
if len(set(gnn_manifest_paths)) != 8 or len(set(dg_manifest_paths)) != 3: raise RuntimeError('FAIL manifest path duplication')
if any('test' in path.name.lower() for path in gnn_manifest_paths + dg_manifest_paths): raise RuntimeError('FAIL test-split-named input prohibited')
print({'stage': 'manifest_paths', 'status': 'PASS', 'gnn_count': 8, 'dg_count': 3, 'explicit_missing_original_gnn': {'adjacency_mode': 'topology', 'seed': 42}, 'input_kind': 'run.json only'})

In [ ]:
comparison_dir = STAGE / 'comparison'
cli = [sys.executable, str(COMPARE)]
for path in gnn_manifest_paths: cli.extend(['--gnn-run-json', str(path)])
for path in dg_manifest_paths: cli.extend(['--dg-run-json', str(path)])
cli.extend(['--allow-incomplete-gnn-matrix', '--output-directory', str(comparison_dir)])
run = subprocess.run(cli, text=True, capture_output=True, check=False)
print(run.stdout.strip())
if run.returncode != 0: raise RuntimeError(f'FAIL comparison exited {run.returncode}:\n{run.stderr}')
result = json.loads((comparison_dir / 'comparison.json').read_text(encoding='utf-8'))
if result.get('status') != 'partial' or result.get('missing_gnn_combinations') != [{'model': 'original_gnn', 'adjacency_mode': 'topology', 'seed': 42}]: raise RuntimeError('FAIL report does not record the sole approved missing GNN run')
if result.get('aggregate', {}).get('gnn_topology', {}).get('n') != 2 or any(result['aggregate'][name]['n'] != 3 for name in ('gnn_identity', 'gnn_shuffled', 'dg_stmtl_based')): raise RuntimeError('FAIL method sample counts')
if result.get('test_access') is not False: raise RuntimeError('FAIL test-access declaration')
print({'stage': 'comparison', 'status': 'PASS', 'scope': result['comparison_scope'], 'missing': result['missing_gnn_combinations'], 'counts': {name: value['n'] for name, value in result['aggregate'].items()}, 'training': False, 'inference': False, 'test_access': False})

In [ ]:
drive_output = DRIVE_BASE / f'geant_h96_partial_dg_stmtl_comparison_v1_{SESSION_ID}'
if drive_output.exists() or drive_output.is_symlink(): raise FileExistsError(f'Refusing existing Drive report directory: {drive_output}')
if drive_output.parent != DRIVE_BASE or DRIVE_BASE.is_symlink(): raise RuntimeError('FAIL Drive output path containment')
drive_output.mkdir(parents=False, exist_ok=False)
report_names = ('comparison.json', 'comparison.csv', 'report.md')
copy_hashes = {}
for name in report_names:
    source = comparison_dir / name
    destination = drive_output / name
    if not source.is_file() or destination.exists() or destination.is_symlink(): raise RuntimeError(f'FAIL copy source/destination check: {name}')
    source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
    with source.open('rb') as src, destination.open('xb') as dst: shutil.copyfileobj(src, dst)
    readback_hash = hashlib.sha256(destination.read_bytes()).hexdigest()
    if readback_hash != source_hash: raise RuntimeError(f'FAIL Drive copy read-back hash: {name}')
    copy_hashes[name] = {'sha256': source_hash, 'readback_sha256': readback_hash, 'bytes': destination.stat().st_size}
copy_record = {'status': 'copied_and_readback_verified', 'comparison_status': 'partial', 'missing_original_gnn': {'adjacency_mode': 'topology', 'seed': 42}, 'files': copy_hashes, 'training': False, 'test_access': False}
record_bytes = (json.dumps(copy_record, sort_keys=True, indent=2) + '\n').encode('utf-8')
with (drive_output / 'copy_verification.json').open('xb') as stream: stream.write(record_bytes)
if hashlib.sha256((drive_output / 'copy_verification.json').read_bytes()).digest() != hashlib.sha256(record_bytes).digest(): raise RuntimeError('FAIL copy-record read-back hash')
print({'stage': 'Drive_report_copy', 'status': 'PASS', 'directory': str(drive_output), 'file_hashes': copy_hashes, 'reporting_only': True})
print('Compact validation summary:')
for method, values in result['aggregate'].items():
    print(f"{method}: n={values['n']}, MAE={values['mae']['mean']:.8g} (sample SD={values['mae']['sample_std_ddof1']}), RMSE={values['rmse']['mean']:.8g} (sample SD={values['rmse']['sample_std_ddof1']})")